# Домашнее задание 5: LLM и RAG

В данной работе вам предстоит создать чат-бота-врача, используя метод Retrieval-Augmented Generation (RAG) и фреймворки Huggingface и LangChain.

---
**Содержание:**
- Загрузка и подготовка данных MEDAL.
- Чтение и индексация данных.
- Создание эмбеддингов и векторного хранилища.
- Построение LLM и настройка поиска.
- Разработка шаблона prompt (Prompt Engineering).
- Создание LangChain Pipeline.
- Бонус: Интеграция истории переписки.

In [ ]:
%pip install datasets langchain_community langchain_chroma langchain langchain_core tiktoken sentence-transformers==2.2.2 lark InstructorEmbedding bitsandbytes accelerate >> /dev/null

## Загрузка датасета MEDAL

В этом разделе мы загружаем датасет [MEDAL](https://huggingface.co/datasets/bigbio/medal), содержащий медицинские статьи для различных клинических диагнозов.

**Замечание:** Нас интересуют колонки `TEXT` и `LABEL`.

Дополнительные ссылки:
- [Репозиторий MEDAL](https://github.com/McGill-NLP/medal)
- [train.csv (Zenodo)](https://zenodo.org/record/4482922/files/train.csv)
- [Файл на Google Drive](https://drive.google.com/file/d/1X7PTIkmsFhTk5n-4W6SWa7XWsDGTpafl/view?usp=sharing)


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

### Задание 0: Загрузка данных MEDAL (0.5 балла)

Просмотр содержимого файла с данными:

In [ ]:
!head -n 10 gdrive/MyDrive/HSE/medal_train.csv

## Задание 1: Чтение и индексация данных (2.5 балла)

Здесь необходимо:
- Прочитать данные из CSV файла.
- Создать генератор документов с использованием метода `.lazy_load()`.

In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader

FILE_PATH = None  # TODO: укажите путь
docs = None  # TODO: создайте генератор

### Создание эмбеддингов и векторного хранилища

На данном этапе:
- Определите модель для вычисления эмбеддингов.
- Создайте векторное хранилище для дальнейшего поиска.

In [ ]:
from langchain.embeddings import HuggingFaceInstructEmbeddings
import torch

# Инициализация модели эмбеддингов
emb_model = HuggingFaceInstructEmbeddings(
    model_name=None  # TODO: Укажите название модели
    )  

### Создание векторного хранилища (индекса)

Настройте индекс для эмбеддингов с использованием Chroma.

In [ ]:
from langchain.vectorstores import Chroma

persist_directory = 'DB'

# Создание индекса
vectordb = None  # TODO: Создайте объект Chroma
vectordb.persist()

### Индексация части документов

Для ускорения работы проиндексируйте первые **N_DOCS** документов, так как полный датасет может содержать миллионы записей.

In [ ]:
from tqdm.auto import tqdm

N_DOCS = 2000  # Обработка первых 2000 документов

for i, doc in tqdm(enumerate(docs), total=N_DOCS):
    # TODO: Добавьте код для обработки каждого документа
    pass

vectordb.persist()

### Проверка содержимого каталога с индексом

In [ ]:
!ls -lht DB

## Задание 2: Создание LLM и настройка поиска (2 балла)

В этом разделе:
- Настройте модель LLM для генерации ответов.
- Проверьте работу поиска по индексу.

In [ ]:
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.retrievers.self_query.base import SelfQueryRetriever
import torch

In [ ]:
model_id = None  # TODO: Укажите идентификатор модели 
llm = None  # TODO: Загрузите модель в базовый класс LangChain LLM

# Создание объекта для поиска по индексу
retriever = vectordb.as_retriever()

In [ ]:
# Проверка работы поиска:
retriever.invoke("ceftobiprole bpr")

## Задание 3: Prompt Engineering. Создание Prompt Template (3 балла)

С помощью синтаксиса `jinja2` настройте шаблон для prompt.

In [ ]:
from pprint import pp as pprint
from transformers import AutoTokenizer

tokenizer = None  # TODO: Инициализируйте токенизатор
pprint(tokenizer.chat_template)

Пример использования PromptTemplate:

```python
PromptTemplate(template=tokenizer.chat_template, template_format='jinja2', input_variables=['content'])
```

*Советы*:
- Prompt Templates можно посмотреть на https://github.com/chujiezheng/chat_templates и [replicate.com](www.replicate.com). Например, для LLama 3 они тут:
    - https://replicate.com/meta/meta-llama-3-70b-instruct
    - https://github.com/chujiezheng/chat_templates/blob/main/chat_templates/llama-3-chat.jinja

```
"""LLama 3 template:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""
```

In [ ]:
from langchain.prompts import PromptTemplate

SYSTEM_PROMPT = None  # TODO: Введите системный промпт

USE_HISTORY = False
if USE_HISTORY:
    # BONUS: Задание с историей переписки, см. ниже
    instruction = None  # TODO: задайте инструкцию с использованием истории переписки
    prompt_template = None  # TODO: вставьте шаблон с историей переписки
    prompt = PromptTemplate(input_variables=None,  # TODO: введите переменные
                            template=prompt_template)
else:
    instruction = None  # TODO: задайте инструкцию
    prompt_template = None  # TODO: вставьте шаблон
    prompt = PromptTemplate(input_variables=None,  # TODO: введите переменные
                            template=prompt_template)

print(prompt)

## Задание 4: Создание Chain (LangChain pipeline) (2 балла)

Соберите пайплайн, включающий следующие этапы:
- **Feature Engineering** (Retrieval Augmentation)
- **Препроцессинг** (Prompt Engineering)
- **Модель LLM**
- **Постпроцессинг**

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers.string import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


FEATURE_ENGINEERING_STAGE = None   # TODO: Реализуйте этап feature engineering (retrieval augmentation)
PREPROCESSING_STAGE = None         # TODO: Реализуйте этап препроцессинга (prompt engineering)
POSTPROCESSING_STAGE = None        # TODO: Реализуйте этап постпроцессинга

chain = (
    {FEATURE_ENGINEERING_STAGE} |
    {PREPROCESSING_STAGE} |
    llm |  # Модель
    (POSTPROCESSING_STAGE)
)

**Примеры вызова цепочки:**

In [ ]:
result1 = chain.invoke('How to treat pneumonia?')  # Пример запроса
print(result1)

In [ ]:
result2 = chain.invoke('Tell in details what is ceftobiprole bpr?')  # Пример запроса
print(result2)


## Бонус (2 балла): Добавление истории переписки

**Подсказка:** Используйте `langchain.memory.ConversationBufferMemory` для интеграции истории переписки с ботом.